<a href="https://colab.research.google.com/github/max7082/colab-practice/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
from torch import nn
from torchvision.datasets import MNIST

device = "cuda" if torch.cuda.is_available() else "cpu"
data = MNIST("./data", train=True, download=True)

x = data.data[:2000].float().reshape(2000, 784).to(device) / 255
label = data.targets[:2000].to(device)
y = nn.functional.one_hot(label, num_classes=10).float()

class ScaledReLU(nn.Module):
    def __init__(self, scale):
        super().__init__()
        self.scale = scale

    def forward(self, x):
        return torch.relu(self.scale * x)

settings = [
    ("ReLU", 1, 0.1),
    ("max(0,10x)", 10, 0.1),
    ("max(0,10x), 학습률 1/10", 10, 0.01)
]

max_epochs = 1000
target_error = 0.25

for name, scale, lr in settings:
    torch.manual_seed(44)

    model = nn.Sequential(
        nn.Linear(784, 64),
        ScaledReLU(scale),
        nn.Linear(64, 10)
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for epoch in range(max_epochs + 1):
        output = model(x)
        loss = nn.functional.cross_entropy(output, y)
        error = (output.argmax(1) != label).float().mean().item()

        if error <= target_error:
            print(f"{name}: {epoch} epoch")
            break

        if epoch == max_epochs:
            break

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

ReLU: 29 epoch
max(0,10x): 21 epoch
max(0,10x), 학습률 1/10: 16 epoch
